In [1]:
from langchain_community.document_loaders import PyPDFLoader

In [2]:
loader = PyPDFLoader("Nigeria.pdf")
data = loader.load()

In [3]:
len(data)

63

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
doc = text_splitter.split_documents(data)
print(f"Total number of documents:", len(doc))

Total number of documents: 282


In [5]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

from dotenv import load_dotenv
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


emnbeddings= GoogleGenerativeAIEmbeddings(model="models/embedding-001",
                                          google_api_key=GOOGLE_API_KEY)
vector = emnbeddings.embed_query("hello, world")

In [6]:
vector[:5]

[0.053597815334796906,
 -0.030782219022512436,
 -0.03252851590514183,
 -0.02810630202293396,
 0.022345904260873795]

In [7]:
vectorstore = Chroma.from_documents(documents = doc,
                                    embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001"))


In [9]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":10})
retrieved_docs = retriever.invoke("What is new in Nigeria?")

In [10]:
print(retrieved_docs[5].page_content)

196. "Nigeria now generates 13,000mw of power, says Minister – Chukwuma" (https://naijalitz.co
m/nigeria-now-generates-13000mw-of-power-says-minister/). Naijalitz – No 1 Entertainment
Portal. Retrieved 28 October 2020.
197. "A new car assembly plant begins operation in Nigeria" (https://www.ntu.edu.sg/cas/news-ev
ents/news/details/a-new-car-assembly-plant-begins-operation-in-nigeria). NTU-SBF Centre
for African Studies (CAS). Archived (https://web.archive.org/web/20220704124830/https://w
ww.ntu.edu.sg/cas/news-events/news/details/a-new-car-assembly-plant-begins-operation-in
-nigeria) from the original on 4 July 2022. Retrieved 30 May 2022.
198. Yager, Thomas R. (March 2022). "The Mineral Industry of Nigeria" (https://pubs.usgs.gov/my
b/vol3/2017-18/myb3-2017-18-nigeria.pdf) (PDF). Archived (https://web.archive.org/web/202
20610111629/https://pubs.usgs.gov/myb/vol3/2017-18/myb3-2017-18-nigeria.pdf) (PDF)
from the original on 10 June 2022. Retrieved 10 June 2022.


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0.3, max_tokens=500)

In [12]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [13]:
system_prompt =(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompts = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [14]:
question_answer_chain = create_stuff_documents_chain(llm,prompts)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [15]:
response = rag_chain.invoke({"input":"What has been the most progressive project in Nigeria"})
print(response["answer"])

Determining the *most* progressive project is subjective and depends on individual priorities. However, the Dangote Refinery, completed in 2022, is the largest refinery south of the Sahara and is expected to significantly impact Nigeria's economy.  Additionally, the Second Niger Bridge, largely completed in 2022, represents a major infrastructure improvement.
